In [1]:
import os
import nbtlib
import sys
from pathlib import Path

In [11]:

def remove_air_only_structures(directory):
    """
    Recursively checks all .nbt files in the given directory.
    Deletes files that represent Minecraft structures composed only of air blocks.
    """
    def get_tag(data, *keys):
        for key in keys:
            if key in data:
                return data[key]
        return None

    no_list = 0
    empty_list = 0
    no_palette = 0
    air_count = 0

    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.nbt'):
                path = os.path.join(root, file)
                try:
                    nbt = nbtlib.load(path)
                    palette = get_tag(nbt, 'palette', 'Palette')
                    blocks = get_tag(nbt, 'blocks', 'Blocks')

                    if blocks is None:
                        # No block list means the structure is empty
                        os.remove(path)
                        no_list += 1
                        continue

                    if len(blocks) == 0:
                        # Empty block list means air-only structure
                        os.remove(path)
                        empty_list += 1
                        continue

                    if palette is None:
                        # If there's a block list but no palette, treat as empty or malformed
                        os.remove(path)
                        no_palette += 1
                        continue

                    air_only = True
                    for block in blocks:
                        state = int(block['state'])
                        name = palette[state]['Name']
                        if name != 'minecraft:air':
                            air_only = False
                            break
                    if air_only:
                        os.remove(path)
                        air_count += 1
                        print(f"Deleted air-only structure: {path}")
                except Exception as e:
                    print(f"Error processing {path}: {e}")
    print(f"no_list: {no_list}, empty_list: {empty_list}, no_palette: {no_palette}, air_count: {air_count}")



In [12]:
# Example usage:
# remove_air_only_structures('/path/to/directory')
remove_air_only_structures(r"C:\Users\Public\PrismLauncher\instances\26.1.2\.minecraft\saves\Cell Museum (May 18, 2026) Editing\datapacks\Celledit\data\celledit\structure\martini3")

no_list: 0, empty_list: 2161, no_palette: 0, air_count: 0
